# Analiza roszczeń wydatków

Ten notatnik demonstruje, jak tworzyć agentów korzystających z wtyczek do przetwarzania wydatków podróżnych na podstawie lokalnych obrazów paragonów, generowania e-maila z roszczeniem wydatków oraz wizualizacji danych dotyczących wydatków za pomocą wykresu kołowego. Agenci dynamicznie wybierają funkcje w zależności od kontekstu zadania.

Kroki:
1. Agent OCR przetwarza lokalny obraz paragonu i wyciąga dane dotyczące wydatków podróżnych.
2. Agent e-mail generuje e-mail z roszczeniem wydatków.

### Przykład scenariusza wydatków podróżnych:
Wyobraź sobie, że jesteś pracownikiem podróżującym na spotkanie biznesowe do innego miasta. Twoja firma ma politykę zwrotu wszystkich uzasadnionych wydatków związanych z podróżą. Oto podział potencjalnych wydatków podróżnych:
- Transport:
Bilet lotniczy na podróż w obie strony z miasta zamieszkania do miasta docelowego.  
Taksówka lub usługi przewozu na lotnisko i z lotniska.  
Lokalny transport w mieście docelowym (np. transport publiczny, wynajem samochodów lub taksówki).

- Zakwaterowanie:
Pobyt w hotelu przez trzy noce w średniej klasy hotelu biznesowym blisko miejsca spotkania.

- Posiłki:
Dzienny dodatek na posiłki na śniadanie, lunch i kolację, zgodnie z polityką firmy dotyczącą diety.

- Wydatki różne:
Opłaty za parking na lotnisku.  
Opłaty za dostęp do Internetu w hotelu.  
Napiwki lub drobne opłaty serwisowe.

- Dokumentacja:
Przekazujesz wszystkie paragony (za loty, taksówki, hotel, posiłki itp.) oraz wypełniony raport wydatków do zwrotu.


## Importuj wymagane biblioteki

Zaimportuj niezbędne biblioteki i moduły do notatnika.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Define Expense Models

 Utwórz model Pydantic dla poszczególnych wydatków oraz klasę ExpenseFormatter do konwersji zapytania użytkownika na ustrukturyzowane dane o wydatkach.

 Każdy wydatek będzie reprezentowany w formacie:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Defining Tools - Generowanie e-maila

Utwórz funkcję narzędzia do generowania e-maila do zgłoszenia roszczenia kosztów.
- To narzędzie używa dekoratora `@tool` z Microsoft Agent Framework.
- Oblicza całkowitą kwotę wydatków i formatuje szczegóły w treści e-maila.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Narzędzie do wyodrębniania kosztów podróży z obrazów paragonów

Utwórz funkcję narzędzia do wyodrębniania kosztów podróży z obrazów paragonów.
- To narzędzie korzysta z dekoratora `@tool` z Microsoft Agent Framework.
- Odczytuje obraz paragonu, koduje go jako base64 i zwraca URI danych do analizy przez agenta.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Przetwarzanie wydatków

Zdefiniuj agentów i połącz ich w sekwencyjny proces za pomocą `WorkflowBuilder`.
- Agent OCR wyodrębnia ustrukturyzowane dane dotyczące wydatków z obrazu paragonu, korzystając z narzędzia `load_receipt_image`.
- Agent Email pobiera wyodrębnione dane i generuje profesjonalny e-mail z roszczeniem wydatkowym, korzystając z narzędzia `generate_expense_email`.
- `WorkflowBuilder` z `add_edge` tworzy sekwencyjną linię przetwarzania: Agent OCR → Agent Email.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Main function

Zbuduj sekwencyjny przepływ pracy i uruchom go, aby przetworzyć obraz paragonu oraz wygenerować e-mail z rozliczeniem wydatków.


> **Uwaga:** Ten przepływ pracy obecnie przekazuje obraz paragonu jako tekst base64, którego większość modeli czatu (w tym gpt-4o) nie będzie traktować jako obraz.
> Może to także przekroczyć okno kontekstu modelu. Zaleca się uruchomienie OCR za pomocą Azure AI Vision (lub innego narzędzia OCR) i przekazanie tylko wyodrębnionego tekstu lub przekształcenie tak, aby wysyłać obraz jako komunikat `image_url`.
> Jeśli chcesz uniknąć błędów kontekstu, spróbuj mniejszego obrazu paragonu lub modelu z większym oknem kontekstu.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
